# 1. Setup

**Zweck dieses Notebooks:** die (teure) Clique-Discovery -- für jeden Tag und Basket die aktuell persistenteste Clique per `HelperFunctions.get_persistent_clusters` finden -- EINMAL für den gesamten verfügbaren Zeitraum ausführen und als Parquet-Datei ablegen (`data/processed/cliques/`). Nachgelagerte Notebooks (z.B. `03_signal_generation/01_forward_return_screening.ipynb`, später die Backtest-Engine) lesen diese Datei ein statt die Discovery selbst zu wiederholen.

Nutzt dieselben kalibrierten Parameter wie `03_pointclouds.ipynb` (roh) und `04_residual_pointclouds.ipynb` (residualisiert) -- Abschnitt 4 dort. **Wichtig:** wird dort neu kalibriert, muss dieses Notebook erneut ausgeführt werden, sonst ist der Cache veraltet (die kalibrierten Werte werden mit in die Cache-Metadaten geschrieben, damit lesende Notebooks das prüfen können, siehe Abschnitt 6).

In [10]:
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))
sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import pandas as pd

from config import (
    ASSET_DAILY_PATH,
    ASSET_ID_COL,
    ASSETS_PATH,
    CLOSE_COL,
    DATE_COL,
    MARKET_SYMBOL_COL,
    MIN_VOLUME_24H_USD,
    OHLCV_DIR,
    PROCESSED_DIR,
    PROJECT_ROOT,
    TIMESTAMP_COL,
    VOLUME_COL,
)
from helperfunctions import HelperFunctions, STABLECOIN_SYMBOLS

helper = HelperFunctions()

print("Imports OK")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Imports OK


# 2. Daten laden & Basket-Konfiguration

In [11]:
# --- Daten laden & vorverarbeiten (identisch zu 03_pointclouds.ipynb) ---
assets_df = pd.read_parquet(ASSETS_PATH)[["asset_id", "symbol"]].drop_duplicates()
id_to_symbol = dict(zip(assets_df["asset_id"], assets_df["symbol"]))
sym_to_id = dict(zip(assets_df["symbol"], assets_df["asset_id"]))

excluded_ids_stablecoins: set[int] = set(
    assets_df.loc[assets_df["symbol"].isin(STABLECOIN_SYMBOLS), "asset_id"].tolist()
)

ohlcv_files = sorted(OHLCV_DIR.glob("*.parquet"))
raw = pd.concat([pd.read_parquet(f) for f in ohlcv_files], ignore_index=True)
raw["base_symbol"] = raw[MARKET_SYMBOL_COL].str.replace("USDT$", "", regex=True)
raw = raw.merge(assets_df.rename(columns={"symbol": "base_symbol"}), on="base_symbol", how="left")
raw = raw.dropna(subset=["asset_id"])
raw["asset_id"] = raw["asset_id"].astype(int)
raw[TIMESTAMP_COL] = pd.to_datetime(raw[TIMESTAMP_COL], utc=True)
raw[DATE_COL] = raw[TIMESTAMP_COL].dt.normalize()
ohlcv_df = raw[[TIMESTAMP_COL, DATE_COL, ASSET_ID_COL, CLOSE_COL]].copy()

data_min = ohlcv_df[DATE_COL].min().date()
data_max = ohlcv_df[DATE_COL].max().date()

universe_df = pd.read_parquet(ASSET_DAILY_PATH)
universe_df[DATE_COL] = pd.to_datetime(universe_df[DATE_COL], utc=True)
all_asset_ids_local = set(universe_df[ASSET_ID_COL].unique())

print(f"Zeitraum: {data_min} \u2192 {data_max}")

Zeitraum: 2024-07-23 → 2026-07-22


In [12]:
# --- Basket-Konfiguration (identisch zu 03_pointclouds.ipynb) ---
POINTCLOUD_LOOKBACK_DAYS = 7
POINTCLOUD_BASKET_UPPER_BOUNDS = [100_000_000, 250_000_000, 1_000_000_000]

rolling_volume_df = helper.compute_rolling_volume_table(
    universe_df, id_column=ASSET_ID_COL, date_column=DATE_COL,
    volume_column=VOLUME_COL, lookback_days=POINTCLOUD_LOOKBACK_DAYS,
)
basket_labels = [f"Basket {n}" for n in range(1, len(POINTCLOUD_BASKET_UPPER_BOUNDS) + 2)]

# 3. Kalibrierte Parameter (aus 02_tda_clustering übernommen)

Direkt aus `03_pointclouds.ipynb` (roh) und `04_residual_pointclouds.ipynb` (residualisiert) Abschnitt 4 kopiert -- gleiches Muster wie überall sonst, wo diese Werte gebraucht werden.

In [13]:
# --- Roh (03_pointclouds.ipynb, Abschnitt 4) ---
BEST_LOOKBACK_RAW: dict[str, int] = {
    "Basket 1": 50,
    "Basket 2": 55,
    "Basket 3": 20,
    "Basket 4": 20,
}
BEST_THRESHOLD_RAW: dict[str, float] = {
    "Basket 1": 0.2175,
    "Basket 2": 0.1028,
    "Basket 3": 0.1260,
    "Basket 4": 0.2636,
}
EPSILON_CAP_RAW = 1.0

# --- Residualisiert (04_residual_pointclouds.ipynb, Abschnitt 4) ---
FACTOR_COLUMNS_RESIDUAL = ["BTC", "ETH"]
REGRESSION_LOOKBACK_DAYS = 60
BEST_LOOKBACK_RESIDUAL: dict[str, int] = {
    "Basket 1": 10,
    "Basket 2": 30,
    "Basket 3": 35,
    "Basket 4": 40,
}
BEST_THRESHOLD_RESIDUAL: dict[str, float] = {
    "Basket 1": 0.1599,
    "Basket 2": 0.0849,
    "Basket 3": 0.1025,
    "Basket 4": 0.1783,
}
EPSILON_CAP_RESIDUAL = 1.3

factor_asset_ids: set[int] = {sym_to_id[s] for s in FACTOR_COLUMNS_RESIDUAL if s in sym_to_id}

print("Roh:            Lookback", BEST_LOOKBACK_RAW, "| epsilon_cap", EPSILON_CAP_RAW)
print("Residualisiert: Lookback", BEST_LOOKBACK_RESIDUAL, "| epsilon_cap", EPSILON_CAP_RESIDUAL)

Roh:            Lookback {'Basket 1': 50, 'Basket 2': 55, 'Basket 3': 20, 'Basket 4': 20} | epsilon_cap 1.0
Residualisiert: Lookback {'Basket 1': 10, 'Basket 2': 30, 'Basket 3': 35, 'Basket 4': 40} | epsilon_cap 1.3


# 4. Cliquen berechnen & cachen: roh

**Achtung, das ist die teure Zelle:** volle Persistenzberechnung für JEDEN Tag im gesamten Datenbereich (nicht nur den Kalibrierungszeitraum -- dieser Cache soll auch später für den Backtest wiederverwendbar sein) x 4 Baskets. Läuft nur bei `RUN_DISCOVERY=True`; danach steht das Ergebnis in der Parquet-Datei und muss nicht mehr neu gerechnet werden, bis sich die kalibrierten Parameter oben ändern.

`DATE_STRIDE=1` (jeder einzelne Tag) ist Standard, weil der Cache möglichst vollständig/allgemein wiederverwendbar sein soll -- nachgelagerte Notebooks können beim Einlesen selbst weiter ausdünnen, wenn sie es eiliger haben.

In [14]:
RUN_DISCOVERY = True
DATE_STRIDE = 1  # 1 = jeder Tag; nur zum schnellen Testen hochsetzen

all_dates = pd.date_range(data_min, data_max, freq="D", tz="UTC")[::DATE_STRIDE]

if RUN_DISCOVERY:
    cliques_raw = HelperFunctions.discover_cliques_over_dates(
        ohlcv_df=ohlcv_df, universe_df=universe_df, rolling_volume_df=rolling_volume_df,
        id_column=ASSET_ID_COL, date_column=DATE_COL,
        basket_labels=basket_labels, basket_upper_bounds=POINTCLOUD_BASKET_UPPER_BOUNDS,
        exclude_ids_for_search=excluded_ids_stablecoins, all_asset_ids=all_asset_ids_local,
        best_lookback=BEST_LOOKBACK_RAW, best_threshold=BEST_THRESHOLD_RAW, epsilon_cap=EPSILON_CAP_RAW,
        point_cloud_fn=HelperFunctions.build_point_cloud,
        dates=all_dates,
    )
    print(f"cliques_raw: {len(cliques_raw)} gefundene Cliquen über {len(all_dates)} Tage x {len(basket_labels)} Baskets")
else:
    print("RUN_DISCOVERY=False -- cliques_raw nicht neu berechnet.")

cliques_raw: 1750 gefundene Cliquen über 730 Tage x 4 Baskets


In [15]:
CLIQUES_DIR = PROCESSED_DIR / "cliques"
CLIQUES_DIR.mkdir(parents=True, exist_ok=True)

cliques_raw.to_parquet(CLIQUES_DIR / "cliques_raw.parquet", index=False)
json.dump(
    {
        "variant": "raw",
        "best_lookback": BEST_LOOKBACK_RAW,
        "best_threshold": BEST_THRESHOLD_RAW,
        "epsilon_cap": EPSILON_CAP_RAW,
        "basket_upper_bounds": POINTCLOUD_BASKET_UPPER_BOUNDS,
        "date_range": [str(data_min), str(data_max)],
        "date_stride": DATE_STRIDE,
    },
    open(CLIQUES_DIR / "cliques_raw.meta.json", "w"),
    indent=2,
)
print(f"Gespeichert: {(CLIQUES_DIR / 'cliques_raw.parquet').relative_to(PROJECT_ROOT)}")

Gespeichert: data/processed/cliques/cliques_raw.parquet


# 5. Cliquen berechnen & cachen: residualisiert

`_residual_point_cloud_fn` übersetzt zwischen der `point_cloud_fn`-Signatur und `HelperFunctions.build_residual_point_cloud` -- identisches Wrapper-Muster wie in `04_residual_pointclouds.ipynb` Abschnitt 3.

In [16]:
def _residual_point_cloud_fn(*, ohlcv_df, universe_df, end_date, window_size_days, exclude_asset_ids):
    return HelperFunctions.build_residual_point_cloud(
        ohlcv_df=ohlcv_df, universe_df=universe_df, end_date=end_date,
        id_to_symbol=id_to_symbol, factor_columns=FACTOR_COLUMNS_RESIDUAL,
        regression_lookback_days=REGRESSION_LOOKBACK_DAYS,
        topological_lookback_days=window_size_days,
        exclude_asset_ids=exclude_asset_ids,
    )


if RUN_DISCOVERY:
    cliques_residual = HelperFunctions.discover_cliques_over_dates(
        ohlcv_df=ohlcv_df, universe_df=universe_df, rolling_volume_df=rolling_volume_df,
        id_column=ASSET_ID_COL, date_column=DATE_COL,
        basket_labels=basket_labels, basket_upper_bounds=POINTCLOUD_BASKET_UPPER_BOUNDS,
        exclude_ids_for_search=excluded_ids_stablecoins | factor_asset_ids, all_asset_ids=all_asset_ids_local,
        best_lookback=BEST_LOOKBACK_RESIDUAL, best_threshold=BEST_THRESHOLD_RESIDUAL, epsilon_cap=EPSILON_CAP_RESIDUAL,
        point_cloud_fn=_residual_point_cloud_fn,
        dates=all_dates,
    )
    print(f"cliques_residual: {len(cliques_residual)} gefundene Cliquen über {len(all_dates)} Tage x {len(basket_labels)} Baskets")
else:
    print("RUN_DISCOVERY=False -- cliques_residual nicht neu berechnet.")

cliques_residual: 1188 gefundene Cliquen über 730 Tage x 4 Baskets


In [17]:
cliques_residual.to_parquet(CLIQUES_DIR / "cliques_residual.parquet", index=False)
json.dump(
    {
        "variant": "residual",
        "factor_columns": FACTOR_COLUMNS_RESIDUAL,
        "regression_lookback_days": REGRESSION_LOOKBACK_DAYS,
        "best_lookback": BEST_LOOKBACK_RESIDUAL,
        "best_threshold": BEST_THRESHOLD_RESIDUAL,
        "epsilon_cap": EPSILON_CAP_RESIDUAL,
        "basket_upper_bounds": POINTCLOUD_BASKET_UPPER_BOUNDS,
        "date_range": [str(data_min), str(data_max)],
        "date_stride": DATE_STRIDE,
    },
    open(CLIQUES_DIR / "cliques_residual.meta.json", "w"),
    indent=2,
)
print(f"Gespeichert: {(CLIQUES_DIR / 'cliques_residual.parquet').relative_to(PROJECT_ROOT)}")

Gespeichert: data/processed/cliques/cliques_residual.parquet


# 6. Kurzer Überblick

In [18]:
for name, df in [("roh", cliques_raw), ("residualisiert", cliques_residual)]:
    print(f"--- {name} ---")
    print(f"  {len(df)} Zeilen, {df['discovery_date'].nunique()} Tage mit mind. einer Clique")
    print(f"  Cliquen pro Basket:\n{df['basket'].value_counts().sort_index().to_string()}")
    print(f"  Mittlere Cliquen-Grösse: {df['size'].mean():.2f}, mittlere Persistenz: {df['persistence'].mean():.4f}")
    print()

--- roh ---
  1750 Zeilen, 674 Tage mit mind. einer Clique
  Cliquen pro Basket:
basket
Basket 1    503
Basket 2    598
Basket 3    458
Basket 4    191
  Mittlere Cliquen-Grösse: 7.79, mittlere Persistenz: 0.2549

--- residualisiert ---
  1188 Zeilen, 596 Tage mit mind. einer Clique
  Cliquen pro Basket:
basket
Basket 1    235
Basket 2    440
Basket 3    354
Basket 4    159
  Mittlere Cliquen-Grösse: 7.38, mittlere Persistenz: 0.2021

